# Figure 3: PW-ADM1 validation

Run from the `notebooks` folder. No Brightway needed. Writes `results/validation/fig3_results.csv` and `figures/Figure3A_bars.png`, `figures/Figure3B_parity.png`.

In [ ]:
# =============================================================================
# Figure 3: PW-ADM1 and standard ADM1 vs measured BMP of HTC process water
#   Four studies, ten HTC conditions, batch mode. Chain per point: HTC
#   surrogate -> HTC mass balance -> process water elements by difference
#   (feed - char - gas, 1 kg dry feed) -> translate_pw_to_adm1 (batch, the
#   study's ISR and COD) -> blank-corrected batch ADM1 run.
#   Standard ADM1: same inert COD in the single S_I pool, no sub-pools.
# =============================================================================
import os, sys, importlib.util, contextlib, io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

P1 = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "ADM1")) else os.path.dirname(os.getcwd())   # repository root
MC_CHAR = 0.35            # filter cake moisture, as in the case study

# ---------------------------------------------------------------- data
# feed: ash, VM, FC, C, H, N, S (% dry); T (C), RT (min), SL (% solids in HTC), batch duration (d),
# COD in the BMP test (g/L), ISR; exp = measured BMP of the process water, mL CH4 (STP)/g COD
cols = ["study", "feed", "ash", "vm", "fc", "c", "h", "n", "s", "T", "RT", "SL", "dur", "COD", "ISR", "exp"]
pts = pd.DataFrame([
    ["Aragón-Briceño et al. (2017)", "Sewage digestate", 31.77, 55.81, 7.76, 34.78, 3.93, 5.04, 1.15, 160, 30, 4.5, 21, 12.6, 0.8, 260],
    ["Aragón-Briceño et al. (2017)", "Sewage digestate", 31.77, 55.81, 7.76, 34.78, 3.93, 5.04, 1.15, 220, 30, 4.5, 21, 13.0, 0.8, 277.2],
    ["Aragón-Briceño et al. (2017)", "Sewage digestate", 31.77, 55.81, 7.76, 34.78, 3.93, 5.04, 1.15, 250, 30, 4.5, 21, 12.1, 0.8, 225.8],
    ["Lucian et al. (2020)", "OFMSW", 17.9, 71.3, 10.8, 42.9, 6.4, 2.8, 0.0, 180, 60, 12.4, 35, 6.5, 2, 205],
    ["Lucian et al. (2020)", "OFMSW", 17.9, 71.3, 10.8, 42.9, 6.4, 2.8, 0.0, 220, 180, 12.4, 35, 6.5, 2, 185],
    ["Lucian et al. (2020)", "OFMSW", 17.9, 71.3, 10.8, 42.9, 6.4, 2.8, 0.0, 250, 360, 12.4, 35, 6.5, 2, 166],
    ["Periyavaram et al. (2023)", "FW", 9.1, 82.63, 8.3, 38.25, 5.47, 3.66, 0.5, 198.5, 150, 16.7, 30, 5, 2, 220],
    ["Erdogan et al. (2015)", "Orange pomace", 5.5, 81, 13.5, 45.4, 6.3, 1.59, 0.15, 190, 120, 11.1, 14, 2, 2, 213],
    ["Erdogan et al. (2015)", "Orange pomace", 5.5, 81, 13.5, 45.4, 6.3, 1.59, 0.15, 225, 120, 11.1, 14, 2, 2, 209],
    ["Erdogan et al. (2015)", "Orange pomace", 5.5, 81, 13.5, 45.4, 6.3, 1.59, 0.15, 260, 120, 11.1, 14, 2, 2, 195],
], columns=cols)

# ---------------------------------------------------------------- load HADES
def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m; spec.loader.exec_module(m)
    return m

sb = _load("sb_fig3", os.path.join(P1, "ADM1", "system_builder.py"))
sb.plot_adm1_results = lambda **kw: None
MLDIR = os.path.join(P1, "ML training scripts and models")
ml = _load("ml_fig3", os.path.join(MLDIR, "htc_surrogate.py"))
predict = lambda feed: sb.run_ml_model(ml.HTC_parametric_ML, feed)[1]


def bmp(r, pw_mode):
    """Blank-corrected batch BMP of the HTC process water, mL CH4 (STP) per g COD."""
    T, SL = float(r["T"]), r["SL"] / 100.0
    O = 100.0 - r["c"] - r["h"] - r["n"] - r["s"] - r["ash"]
    feed = dict(T=T, RT=r["RT"], WC=100.0 - r["SL"], VM_feed=r["vm"], FC_feed=r["fc"], C_feed=r["c"],
                H_feed=r["h"], N_feed=r["n"], O_feed=O, S_feed=r["s"], Ash_feed=r["ash"])
    with contextlib.redirect_stdout(io.StringIO()):
        inv = predict(feed)
        mb = sb.compute_htc_mass_balance(1.0 / SL, 1 - SL, inv["Y_char (%)"], T, MC_CHAR, 0.10)   # 1 kg dry feed
        Y = inv["Y_char (%)"] / 100.0
        el = {"C": r["c"], "H": r["h"], "O": O, "N": r["n"], "S": r["s"]}
        ch = {e: Y * inv[f"{e}_char (%)"] / 100.0 for e in el}
        C_gas = (mb["CO2_yield"] / sb.MW_CO2 + mb["CO_yield"] / sb.MW_CO) * sb.MW_C
        O_gas = mb["CO2_yield"] * 2 * sb.MW_O / sb.MW_CO2 + mb["CO_yield"] * sb.MW_O / sb.MW_CO
        pw = {"C": el["C"] / 100 - ch["C"] - C_gas, "H": el["H"] / 100 - ch["H"],
              "O": el["O"] / 100 - ch["O"] - O_gas, "N": el["N"] / 100 - ch["N"], "S": el["S"] / 100 - ch["S"]}
        pw = {k: max(0.0, v) for k, v in pw.items()}
        d = sb.translate_pw_to_adm1(C_pw=pw["C"], H_pw=pw["H"], O_pw=pw["O"], N_pw=pw["N"], S_pw=pw["S"],
                                    PW_yield_L=mb["PW_yield"] / 1.02, mode="batch", T_HTC=T,
                                    RT_HTC=r["RT"], ISR_batch=r["ISR"], COD_conc_target=r["COD"])
        if not pw_mode:   # ADM1 without extension: same inert COD in the single S_I pool
            names = sb._adm1_core.SI_SUBPOOL_NAMES
            si = sum(d.get(f"S_I_{sp}", 0.0) for sp in names)
            for sp in names:
                d[f"S_I_{sp}"] = d[f"_inf_S_I_{sp}"] = 0.0
            d["S_I"] = d["_inf_S_I"] = si
            d["_pw_mode"] = False
        CH4, _, _, out, _ = sb.run_adm1_and_correct(d, "batch", None, None, float(r["dur"]), T, r["RT"],
                                                    ISR_batch=r["ISR"], label="fig3")
    mL = CH4 / sb.CH4_density * 1e6
    return dict(BMP=mL / (d["_COD_conc"] * out["_V_liq"] * 1e3),     # per g COD loaded in the reactor
                pH_end=float(out["simulate_results"]["pH"].iloc[-1]))


rows = []
for i, r in pts.iterrows():
    a, b = bmp(r, True), bmp(r, False)
    rows.append({**r.to_dict(), "PW": a["BMP"], "ADM1": b["BMP"], "pH_end_PW": a["pH_end"], "pH_end_ADM1": b["pH_end"]})
    print(f"{r['study'][:22]:22s} {r['T']:5.1f} C | exp {r['exp']:5.1f} | PW-ADM1 {a['BMP']:5.1f} "
          f"| ADM1 {b['BMP']:5.1f} | pH {a['pH_end']:.2f}")
res = pd.DataFrame(rows)
res.to_csv(os.path.join(P1, "results", "validation", "fig3_results.csv"), index=False)

def stats(m, p):
    m, p = np.asarray(m, float), np.asarray(p, float); e = p - m
    return f"R² = {1 - (e**2).sum() / ((m - m.mean())**2).sum():.2f}, RMSE = {np.sqrt((e**2).mean()):.1f}, bias = {e.mean():+.1f}"
print("\nPW-ADM1:", stats(res.exp, res.PW), "\nADM1   :", stats(res.exp, res.ADM1))
for k in ("PW", "ADM1"):                                   # quoted in the manuscript text
    n20 = int(((res[k] / res.exp - 1).abs() <= 0.20).sum())
    print(f"{k}: {n20} of {len(res)} predictions within +/-20 % of the measured BMP")

# ---------------------------------------------------------------- figures
COL = {"Aragón-Briceño et al. (2017)": "#4a6fa5", "Lucian et al. (2020)": "#6a994e",
       "Periyavaram et al. (2023)": "#d4a72c", "Erdogan et al. (2015)": "#c8744a"}
fig, ax = plt.subplots(figsize=(14, 4.2)); x, ticks, labs, w = 0.0, [], [], 0.16
for st, g in res.groupby("study", sort=False):
    x0 = x
    for _, r in g.iterrows():
        for j, (v, hatch) in enumerate([(r.exp, None), (r.PW, "////"), (r.ADM1, "....")]):
            ax.bar(x + (j - 1) * w, v, w, color=COL[st], edgecolor="black", lw=0.6, hatch=hatch)
        ticks.append(x); labs.append(f"{r['T']:.0f}°C"); x += 1
    ax.text((x0 + x - 1) / 2, 352, st, ha="center", fontweight="bold")
    ax.text((x0 + x - 1) / 2, 338, f"({g.feed.iloc[0]})", ha="center", style="italic", color="#555")
    ax.axvline(x - 0.25, color="#ccc", lw=0.8); x += 0.6
ax.set_xticks(ticks); ax.set_xticklabels(labs); ax.set_ylim(0, 350); ax.set_ylabel("BMP (mL CH$_4$/g COD)")
from matplotlib.patches import Patch
ax.legend([Patch(fc="grey", ec="k"), Patch(fc="grey", ec="k", hatch="////"), Patch(fc="grey", ec="k", hatch="....")],
          ["Experimental", "PW-ADM1 (Extension)", "ADM1 (No Extension)"], ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.12), frameon=True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.savefig(os.path.join(P1, "figures", "Figure3A_bars.png"), dpi=600, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5.5)); lim = [100, 360]
ax.fill_between(lim, [v * 0.8 for v in lim], [v * 1.2 for v in lim], color="#ececec")
ax.plot(lim, lim, "k-", lw=1.2, label="1:1 line")
for st, g in res.groupby("study", sort=False):
    ax.scatter(g.exp, g.PW, s=60, color=COL[st], label=st)
    ax.scatter(g.exp, g.ADM1, s=60, marker="D", facecolor="white", edgecolor=COL[st])
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
ax.set_xlabel("Experimental BMP (mL CH$_4$/g COD)"); ax.set_ylabel("Simulated BMP (mL CH$_4$/g COD)")
ax.legend(loc="lower right", fontsize=8, title="● PW-ADM1   ◇ ADM1")
fig.savefig(os.path.join(P1, "figures", "Figure3B_parity.png"), dpi=600, bbox_inches="tight")
plt.show()